# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAli128106/Week-1_A1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

df_features = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_avg_position,
        gsc_clicks,
        dayofweek(report_date) AS day_of_week,
        gsc_clicks * 1.0 / gsc_impressions AS ctr
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
""").df()

df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,day_of_week,ctr
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,0,0,0.000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,0,0,0.000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,1,0,0.008
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,0,0,0.000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,0,0,0.000


In [4]:
df_features.isnull().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
gsc_impressions,0
gsc_avg_position,0
gsc_clicks,0
day_of_week,0
ctr,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


- **gsc_impressions**: daily search impressions. No missing values. Available before CTR is computed " GSC's own output, not click-derived.
- **gsc_avg_position** : daily avg ranking. No missing values. Available same as impressions — ranking, not clicks.
- **day_of_week** — 0–6 from **report_date**. No missing values, always computable in advance.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# rebuild feature frame with gsc_clicks included for the leakage attack
df_features = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_avg_position,
        gsc_clicks,
        dayofweek(report_date) AS day_of_week,
        gsc_clicks * 1.0 / gsc_impressions AS ctr
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
""").df()

y = df_features['ctr']

# Attack 1: label-derived column
X_honest = df_features[['gsc_impressions', 'gsc_avg_position', 'day_of_week']]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
score_honest = r2_score(y_test, DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train).predict(X_test))
print("Honest R²:", score_honest)

X_leaky = df_features[['gsc_impressions', 'gsc_avg_position', 'day_of_week', 'gsc_clicks']]
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
score_leaky = r2_score(y_test, DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train).predict(X_test))
print("Leaky R² (gsc_clicks added):", score_leaky)

print(f"\nJump: {score_honest:.3f} -> {score_leaky:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest R²: 0.0055840850772719275
Leaky R² (gsc_clicks added): 0.9984016710595076

Jump: 0.006 -> 0.998


Label-derived: added `gsc_clicks`. R² 0.006 → 0.998. Removed — leakage, not signal.
Future windows: not applicable as no feature references dates outside the current row.
Product flags: not applicable as all flag columns already excluded in the contract.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


- All GA4 columns : post-click behavior, leaks the outcome.
- gsc_clicks : CTR's numerator, caused the leak above.
- Boolean flags : constant after filtering, no signal.
- gsc_sum_position : redundant with gsc_avg_position.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.